# Neuro-CX — Exploratory Data Analysis

Explores the raw Ecommerce Consumer Behavior dataset and the synthesised interaction sequences.


In [ ]:
import sys, os
os.chdir('..')   # run from neuro-cx/
sys.path.insert(0, 'src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = '#1a1a2e'
plt.rcParams['axes.facecolor']   = '#16213e'
plt.rcParams['axes.edgecolor']   = '#2d3561'
plt.rcParams['text.color']       = '#e0e0e0'
plt.rcParams['axes.labelcolor']  = '#e0e0e0'
plt.rcParams['xtick.color']      = '#aaa'
plt.rcParams['ytick.color']      = '#aaa'
plt.rcParams['axes.titlecolor']  = '#ffffff'
plt.rcParams['axes.grid']        = True
plt.rcParams['grid.alpha']       = 0.2

print('Libraries loaded.')

## 1. Raw Dataset Overview

In [ ]:
df_raw = pd.read_csv('../Ecommerce_Consumer_Behavior_Analysis_Data.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
# Basic stats
print('Purchase Amount (cleaned):')
df_raw['Purchase_Amount_clean'] = df_raw['Purchase_Amount'].str.replace('$','',regex=False).str.replace(',','').str.strip().astype(float)
print(df_raw['Purchase_Amount_clean'].describe())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Raw Dataset Distributions', fontsize=14, fontweight='bold', color='white')

COLORS = ['#667eea','#764ba2','#f093fb','#4ecdc4','#45b7d1','#96ceb4']

# Purchase category
cat_counts = df_raw['Purchase_Category'].str.strip().value_counts()
axes[0,0].barh(cat_counts.index[:8], cat_counts.values[:8], color=COLORS[0])
axes[0,0].set_title('Purchase Categories')
axes[0,0].set_xlabel('Count')

# Purchase intent
intent_counts = df_raw['Purchase_Intent'].value_counts()
axes[0,1].pie(intent_counts.values, labels=intent_counts.index,
              colors=COLORS, autopct='%1.1f%%', startangle=90,
              textprops={'color':'white', 'fontsize':9})
axes[0,1].set_title('Purchase Intent')

# Purchase channel
ch_counts = df_raw['Purchase_Channel'].value_counts()
axes[0,2].bar(ch_counts.index, ch_counts.values, color=COLORS[2])
axes[0,2].set_title('Purchase Channel')
axes[0,2].tick_params(axis='x', rotation=20)

# Age distribution
axes[1,0].hist(df_raw['Age'], bins=20, color=COLORS[3], edgecolor='none', alpha=0.85)
axes[1,0].set_title('Age Distribution')
axes[1,0].set_xlabel('Age')

# Frequency of purchase
axes[1,1].hist(df_raw['Frequency_of_Purchase'], bins=12, color=COLORS[4], edgecolor='none', alpha=0.85)
axes[1,1].set_title('Purchase Frequency')
axes[1,1].set_xlabel('Purchases per year')

# Customer satisfaction
sat_counts = df_raw['Customer_Satisfaction'].value_counts().sort_index()
axes[1,2].bar(sat_counts.index.astype(str), sat_counts.values, color=COLORS[5])
axes[1,2].set_title('Customer Satisfaction (1-10)')
axes[1,2].set_xlabel('Score')

plt.tight_layout()
plt.savefig('logs/eda_raw_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Synthesised Interaction Sequences

In [ ]:
df_events = pd.read_parquet('data/interactions_sample.parquet')
print(f'Events: {len(df_events):,}  |  Customers: {df_events["customer_id"].nunique()}')
df_events.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Synthesised Interaction Sequences', fontsize=14, fontweight='bold', color='white')

# Action distribution
ac = df_events['action_type'].value_counts()
colors_by_action = {'view':'#74b9ff','add_to_cart':'#c77dff','purchase':'#74c69d','review':'#fca46d','bounce':'#f08080'}
bar_colors = [colors_by_action.get(a,'#aaa') for a in ac.index]
axes[0].bar(ac.index, ac.values, color=bar_colors, edgecolor='none')
axes[0].set_title('Action Type Distribution')
axes[0].set_xlabel('Action')
axes[0].tick_params(axis='x', rotation=20)

# Sequence length per customer
seq_lens = df_events.groupby('customer_id').size()
axes[1].hist(seq_lens, bins=15, color='#667eea', edgecolor='none', alpha=0.85)
axes[1].set_title(f'Sequence Length per Customer\n(mean={seq_lens.mean():.1f})')
axes[1].set_xlabel('# interactions')

# Dwell time distribution per action
for action, color in colors_by_action.items():
    d = df_events[df_events['action_type']==action]['dwell_time']
    if len(d) > 0:
        axes[2].hist(d.clip(0,300), bins=30, alpha=0.6, color=color, label=action, edgecolor='none')
axes[2].set_title('Dwell Time by Action (clipped 300s)')
axes[2].set_xlabel('Seconds')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('logs/eda_sequences.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Training Convergence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Training Convergence', fontsize=14, fontweight='bold', color='white')

for ax, name, color, label in [
    (axes[0], 'gru_baseline', '#4C72B0', 'GRU Baseline'),
    (axes[1], 'neuro_cx',    '#DD8452', 'Neuro-CX'),
]:
    log_path = f'logs/{name}_training_log.csv'
    if os.path.exists(log_path):
        log = pd.read_csv(log_path)
        ax.plot(log['epoch'], log['train_loss'], color=color, lw=2, label='Train loss')
        ax.plot(log['epoch'], log['val_loss'], color=color, lw=2, ls='--', alpha=0.7, label='Val loss')
        best_epoch = log.loc[log['val_loss'].idxmin(), 'epoch']
        best_val   = log['val_loss'].min()
        ax.axvline(best_epoch, color='white', lw=1, ls=':', alpha=0.5, label=f'Best epoch ({int(best_epoch)})')
        ax.set_title(f'{label}  (best val={best_val:.4f})')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Cross-entropy loss')
        ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, 'Log not found\nRun py run_train.py first',
                ha='center', va='center', transform=ax.transAxes, color='#aaa')
        ax.set_title(label)

plt.tight_layout()
plt.savefig('logs/eda_convergence.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Final Metric Comparison

In [ ]:
import json
if os.path.exists('logs/comparison.json'):
    with open('logs/comparison.json') as f:
        comp = json.load(f)
    b = comp['baseline']
    n = comp['neurocx']
    metric_keys = [k for k in b if k not in ('model','split','n_eval')]
    df_comp = pd.DataFrame({
        'Metric': metric_keys,
        'GRU Baseline': [b[k] for k in metric_keys],
        'Neuro-CX':     [n[k] for k in metric_keys],
    })
    df_comp['Delta'] = df_comp['Neuro-CX'] - df_comp['GRU Baseline']
    print(df_comp.to_string(index=False, float_format='{:.4f}'.format))
else:
    print('Run py run_eval.py first.')